## 🎯 Learning Objectives
* Deploy an LLM application using a modern serving framework like FastAPI.
* Instrument the deployed LLM application for comprehensive observability (metrics, traces, logs) using OpenTelemetry.
* Configure and simulate alerting based on key performance indicators (KPIs) and error rates.
* Understand the end-to-end process of taking an LLM from development to a monitored production environment.


## Exercise: Deploy, Instrument, and Set Up Alerts for an LLM Application

### Context
In the world of Agentic AI, deploying LLM applications is only the first step. Ensuring their reliability, performance, and maintainability in production requires robust observability and alerting. This exercise challenges you to build a production-ready LLM inference service, focusing on the critical aspects of deployment, instrumentation, and proactive monitoring.

### Task
Your goal is to create a simple FastAPI application that simulates an LLM inference service. This service must be fully instrumented with OpenTelemetry for traces and metrics, utilize structured logging, and include a basic in-memory alerting mechanism.

1.  **Develop a Simple LLM API:** Create a FastAPI application with a single `POST /generate` endpoint. This endpoint will accept a prompt and return a simulated LLM response using the provided `mock_llm_inference` function.
2.  **Instrument for Observability:**
    *   **Tracing:** Use OpenTelemetry to trace the entire request lifecycle, from the API endpoint call down to the `mock_llm_inference` function. Ensure relevant attributes (e.g., prompt, response, latency, errors) are added to spans.
    *   **Metrics:** Collect the following key metrics using OpenTelemetry:
        *   Total number of requests (counter).
        *   Request latency (histogram).
        *   Number of errors (counter).
    *   **Logging:** Implement structured logging within the `/generate` endpoint to record incoming requests, LLM interactions, and any errors, including relevant context.
3.  **Simulate Alerting:** Integrate the provided `AlertingMetricStore` and `check_alerts` function into your application. Ensure that your `/generate` endpoint updates the `AlertingMetricStore` with request outcomes and latencies. Periodically (or on demand), call `check_alerts` to simulate a monitoring system detecting issues like high error rates or elevated latencies.

### Requirements
*   Use `FastAPI` for the web server.
*   Utilize `OpenTelemetry` for all tracing and metric collection.
*   Employ Python's standard `logging` module with a custom JSON formatter for structured logs.
*   Integrate the provided `mock_llm_inference` function, which simulates LLM behavior including occasional failures and variable latency.
*   Ensure the `AlertingMetricStore` is updated correctly and `check_alerts` can detect issues.
*   Your solution should be runnable and demonstrate all required features.

### Evaluation Criteria
*   **Functional Deployment:** The FastAPI application successfully starts and responds to requests.
*   **Comprehensive Tracing:** Traces are generated for each request, showing spans for the API endpoint and the LLM inference, with meaningful attributes.
*   **Accurate Metrics:** OpenTelemetry metrics (request count, latency histogram, error count) are correctly collected and exported (to console in this exercise).
*   **Structured Logging:** Logs are emitted in JSON format with relevant contextual information for each request and error.
*   **Effective Alerting Simulation:** The `AlertingMetricStore` is correctly populated, and the `check_alerts` function accurately identifies and logs alerts based on defined thresholds when issues are simulated (e.g., by increasing `fail_rate` in `mock_llm_inference`).


In [ ]:
import time
import random
import logging
import json
import os
from contextlib import contextmanager

from fastapi import FastAPI, HTTPException, Request, status
from pydantic import BaseModel

# OpenTelemetry imports
from opentelemetry import trace, metrics
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import ConsoleMetricExporter, PeriodicExportingMetricReader
from opentelemetry.semconv.trace import SpanAttributes

# --- 1. Logging Setup ---
# Custom JSON formatter for structured logging
class JsonFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": self.formatTime(record, self.datefmt),
            "level": record.levelname,
            "message": record.getMessage(),
            "name": record.name,
            "pathname": record.pathname,
            "lineno": record.lineno,
        }
        # Add any extra data passed to the log record
        if hasattr(record, 'extra_data'):
            log_entry.update(record.extra_data)
        # Include exception info if present
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.addHandler(handler)

# --- 2. OpenTelemetry Setup ---
# Resource for identifying our service in traces and metrics
resource = Resource.create(attributes={
    "service.name": "llm-inference-service",
    "service.version": "1.0.0",
    "environment": "development"
})

# Tracer Provider: Configures how traces are created and exported
trace_provider = TracerProvider(resource=resource)
# For demonstration, we'll export traces to the console. In production, you'd use OTLPSpanExporter
# to send traces to a collector like Jaeger, Tempo, or DataDog.
span_exporter = ConsoleSpanExporter()
trace_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_global_tracer_provider(trace_provider)
tracer = trace.get_tracer(__name__)

# Meter Provider: Configures how metrics are created and exported
# Metrics will be exported periodically to the console.
metric_reader = PeriodicExportingMetricReader(
    ConsoleMetricExporter(),
    export_interval_millis=5000 # Export metrics every 5 seconds
)
metric_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_global_meter_provider(metric_provider)
meter = metrics.get_meter(__name__)

# --- 3. Mock LLM Inference Function ---
def mock_llm_inference(prompt: str, fail_rate: float = 0.1, avg_latency: float = 0.5) -> str:
    """
    Simulates an LLM inference call with variable latency and occasional failures.
    This function is already instrumented with a basic OpenTelemetry span.
    """
    with tracer.start_as_current_span("mock_llm_inference") as span:
        span.set_attribute("llm.prompt", prompt)
        
        # Simulate latency with a Gaussian distribution
        latency = max(0.1, random.gauss(avg_latency, avg_latency / 3))
        time.sleep(latency)

        # Simulate occasional failures
        if random.random() < fail_rate:
            span.set_attribute(SpanAttributes.EXCEPTION_TYPE, "LLMError")
            span.set_attribute(SpanAttributes.EXCEPTION_MESSAGE, "Simulated LLM inference failure")
            span.set_status(trace.Status(trace.StatusCode.ERROR, "LLM inference failed"))
            raise RuntimeError("Simulated LLM inference failure")

        response = f"Mock LLM response for: '{prompt[:50]}...'"
        span.set_attribute("llm.response", response)
        span.set_attribute("llm.latency_ms", latency * 1000)
        return response

# --- 4. FastAPI Application Setup ---
app = FastAPI(
    title="LLM Inference Service",
    description="A mock LLM service with observability features.",
    version="1.0.0"
)

# Pydantic model for request body validation
class PromptRequest(BaseModel):
    prompt: str

# --- 5. In-memory Metric Aggregation and Alerting (for student to complete) ---
# This simple store is for demonstrating alert logic within the notebook.
# In a real system, alerts would be configured on your monitoring platform (e.g., Prometheus/Grafana)
# which consumes OpenTelemetry metrics. For this exercise, we'll use a simplified in-memory store.
class AlertingMetricStore:
    def __init__(self, window_seconds: int = 60):
        self.request_timestamps = []
        self.error_timestamps = []
        self.latencies = [] # Stores (timestamp, latency_value)
        self.window_seconds = window_seconds # Time window for calculating metrics

    def record_request(self, success: bool, latency: float):
        current_time = time.time()
        self.request_timestamps.append(current_time)
        self.latencies.append((current_time, latency))
        if not success:
            self.error_timestamps.append(current_time)

    def get_window_metrics(self):
        current_time = time.time()
        window_start = current_time - self.window_seconds

        # Filter data to only include entries within the current window
        recent_requests = [t for t in self.request_timestamps if t >= window_start]
        recent_errors = [t for t in self.error_timestamps if t >= window_start]
        recent_latencies = [l for t, l in self.latencies if t >= window_start]

        request_count = len(recent_requests)
        error_count = len(recent_errors)
        avg_latency = sum(recent_latencies) / len(recent_latencies) if recent_latencies else 0

        # Clean up old data to prevent excessive memory usage in a long-running simulation
        self.request_timestamps = [t for t in self.request_timestamps if t >= window_start]
        self.error_timestamps = [t for t in self.error_timestamps if t >= window_start]
        self.latencies = [(t, l) for t, l in self.latencies if t >= window_start]

        return {
            "request_count": request_count,
            "error_count": error_count,
            "avg_latency": avg_latency
        }

alert_metric_store = AlertingMetricStore(window_seconds=30) # Use a 30-second window for alerts

def check_alerts(threshold_error_rate: float = 0.2, threshold_avg_latency: float = 1.0):
    """
    Simulates an alert manager checking current metrics from the in-memory store.
    Logs a warning if thresholds are crossed.
    """
    metrics_in_window = alert_metric_store.get_window_metrics()
    total_requests = metrics_in_window["request_count"]
    error_count = metrics_in_window["error_count"]
    avg_latency = metrics_in_window["avg_latency"]

    if total_requests == 0:
        logger.info("No requests in the current window to check for alerts.", extra_data={"window_s": alert_metric_store.window_seconds})
        return

    error_rate = error_count / total_requests

    alert_triggered = False
    if error_rate > threshold_error_rate:
        logger.warning(
            "ALERT: High Error Rate!",
            extra_data={
                "alert_type": "error_rate",
                "current_error_rate": f"{error_rate:.2f}",
                "threshold": f"{threshold_error_rate:.2f}",
                "total_requests_in_window": total_requests,
                "error_count_in_window": error_count,
                "window_s": alert_metric_store.window_seconds
            }
        )
        alert_triggered = True
    if avg_latency > threshold_avg_latency:
        logger.warning(
            "ALERT: High Average Latency!",
            extra_data={
                "alert_type": "avg_latency",
                "current_avg_latency_s": f"{avg_latency:.2f}",
                "threshold_s": f"{threshold_avg_latency:.2f}",
                "total_requests_in_window": total_requests,
                "window_s": alert_metric_store.window_seconds
            }
        )
        alert_triggered = True

    if not alert_triggered:
        logger.info(
            "All metrics within normal thresholds for the current window.",
            extra_data={
                "error_rate": f"{error_rate:.2f}",
                "avg_latency": f"{avg_latency:.2f}",
                "total_requests_in_window": total_requests,
                "window_s": alert_metric_store.window_seconds
            }
        )

# You can call check_alerts() periodically to see its output
# For example, after running some requests, call it manually:
# check_alerts()


### Your Turn! Implement the LLM Inference Endpoint with Observability

Now it's your turn to complete the FastAPI application. Fill in the missing pieces to achieve the requirements outlined in the task description.

Specifically, you need to:

1.  **Define the `POST /generate` endpoint:** Create a FastAPI endpoint that accepts a `PromptRequest` (defined above).
2.  **Add OpenTelemetry Tracing:** Wrap the entire endpoint logic and the call to `mock_llm_inference` with appropriate OpenTelemetry spans. Add relevant attributes to these spans.
3.  **Add OpenTelemetry Metrics:** Use the `meter` object to record:
    *   An increment to a `request_counter` for every incoming request.
    *   The duration of each request to a `request_latency_histogram`.
    *   An increment to an `error_counter` if the LLM inference fails.
4.  **Integrate Structured Logging:** Use the `logger` object to log the incoming prompt, the LLM response (or error), and the request duration. Ensure these logs include `extra_data` for structured context.
5.  **Update Alerting Metric Store:** Call `alert_metric_store.record_request()` with the success status and latency of each request.

After implementing, you will need to run this file as a FastAPI application (e.g., `uvicorn your_file_name:app --port 8000 --reload`) and test it using `curl` or a tool like Postman/Insomnia. Remember to also call `check_alerts()` periodically to observe the alerting mechanism.

```python
# Your implementation goes here

# Example of how to define a counter and histogram (you'll need to create these globally)
# request_counter = meter.create_counter(
#     "llm_requests_total", description="Total number of LLM inference requests"
# )
# request_latency_histogram = meter.create_histogram(
#     "llm_request_latency_seconds", description="Latency of LLM inference requests in seconds", unit="s"
# )
# error_counter = meter.create_counter(
#     "llm_errors_total", description="Total number of LLM inference errors"
# )

# @app.post("/generate")
# async def generate_text(request: PromptRequest, http_request: Request):
#     start_time = time.perf_counter()
#     success = False
#     response_content = ""
#     status_code = status.HTTP_200_OK
#     error_message = None

#     # ... your implementation ...

#     duration = time.perf_counter() - start_time
#     alert_metric_store.record_request(success, duration)

#     # ... return response ...

```


In [ ]:
import time
import random
import logging
import json
import os
from contextlib import contextmanager

from fastapi import FastAPI, HTTPException, Request, status
from pydantic import BaseModel

# OpenTelemetry imports
from opentelemetry import trace, metrics
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import ConsoleMetricExporter, PeriodicExportingMetricReader
from opentelemetry.semconv.trace import SpanAttributes

# --- 1. Logging Setup ---
class JsonFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": self.formatTime(record, self.datefmt),
            "level": record.levelname,
            "message": record.getMessage(),
            "name": record.name,
            "pathname": record.pathname,
            "lineno": record.lineno,
        }
        if hasattr(record, 'extra_data'):
            log_entry.update(record.extra_data)
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setFormatter(JsonFormatter())
logger.addHandler(handler)

# --- 2. OpenTelemetry Setup ---
resource = Resource.create(attributes={
    "service.name": "llm-inference-service",
    "service.version": "1.0.0",
    "environment": "development"
})

trace_provider = TracerProvider(resource=resource)
span_exporter = ConsoleSpanExporter()
trace_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_global_tracer_provider(trace_provider)
tracer = trace.get_tracer(__name__)

metric_reader = PeriodicExportingMetricReader(
    ConsoleMetricExporter(),
    export_interval_millis=5000
)
metric_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_global_meter_provider(metric_provider)
meter = metrics.get_meter(__name__)

# --- 3. Mock LLM Inference Function ---
def mock_llm_inference(prompt: str, fail_rate: float = 0.1, avg_latency: float = 0.5) -> str:
    with tracer.start_as_current_span("mock_llm_inference") as span:
        span.set_attribute("llm.prompt", prompt)
        latency = max(0.1, random.gauss(avg_latency, avg_latency / 3))
        time.sleep(latency)

        if random.random() < fail_rate:
            span.set_attribute(SpanAttributes.EXCEPTION_TYPE, "LLMError")
            span.set_attribute(SpanAttributes.EXCEPTION_MESSAGE, "Simulated LLM inference failure")
            span.set_status(trace.Status(trace.StatusCode.ERROR, "LLM inference failed"))
            raise RuntimeError("Simulated LLM inference failure")

        response = f"Mock LLM response for: '{prompt[:50]}...'"
        span.set_attribute("llm.response", response)
        span.set_attribute("llm.latency_ms", latency * 1000)
        return response

# --- 4. FastAPI Application Setup ---
app = FastAPI(
    title="LLM Inference Service",
    description="A mock LLM service with observability features.",
    version="1.0.0"
)

class PromptRequest(BaseModel):
    prompt: str

# --- 5. In-memory Metric Aggregation and Alerting ---
class AlertingMetricStore:
    def __init__(self, window_seconds: int = 60):
        self.request_timestamps = []
        self.error_timestamps = []
        self.latencies = []
        self.window_seconds = window_seconds

    def record_request(self, success: bool, latency: float):
        current_time = time.time()
        self.request_timestamps.append(current_time)
        self.latencies.append((current_time, latency))
        if not success:
            self.error_timestamps.append(current_time)

    def get_window_metrics(self):
        current_time = time.time()
        window_start = current_time - self.window_seconds

        recent_requests = [t for t in self.request_timestamps if t >= window_start]
        recent_errors = [t for t in self.error_timestamps if t >= window_start]
        recent_latencies = [l for t, l in self.latencies if t >= window_start]

        request_count = len(recent_requests)
        error_count = len(recent_errors)
        avg_latency = sum(recent_latencies) / len(recent_latencies) if recent_latencies else 0

        self.request_timestamps = [t for t in self.request_timestamps if t >= window_start]
        self.error_timestamps = [t for t in self.error_timestamps if t >= window_start]
        self.latencies = [(t, l) for t, l in self.latencies if t >= window_start]

        return {
            "request_count": request_count,
            "error_count": error_count,
            "avg_latency": avg_latency
        }

alert_metric_store = AlertingMetricStore(window_seconds=30)

def check_alerts(threshold_error_rate: float = 0.2, threshold_avg_latency: float = 1.0):
    metrics_in_window = alert_metric_store.get_window_metrics()
    total_requests = metrics_in_window["request_count"]
    error_count = metrics_in_window["error_count"]
    avg_latency = metrics_in_window["avg_latency"]

    if total_requests == 0:
        logger.info("No requests in the current window to check for alerts.", extra_data={"window_s": alert_metric_store.window_seconds})
        return

    error_rate = error_count / total_requests

    alert_triggered = False
    if error_rate > threshold_error_rate:
        logger.warning(
            "ALERT: High Error Rate!",
            extra_data={
                "alert_type": "error_rate",
                "current_error_rate": f"{error_rate:.2f}",
                "threshold": f"{threshold_error_rate:.2f}",
                "total_requests_in_window": total_requests,
                "error_count_in_window": error_count,
                "window_s": alert_metric_store.window_seconds
            }
        )
        alert_triggered = True
    if avg_latency > threshold_avg_latency:
        logger.warning(
            "ALERT: High Average Latency!",
            extra_data={
                "alert_type": "avg_latency",
                "current_avg_latency_s": f"{avg_latency:.2f}",
                "threshold_s": f"{threshold_avg_latency:.2f}",
                "total_requests_in_window": total_requests,
                "window_s": alert_metric_store.window_seconds
            }
        )
        alert_triggered = True

    if not alert_triggered:
        logger.info(
            "All metrics within normal thresholds for the current window.",
            extra_data={
                "error_rate": f"{error_rate:.2f}",
                "avg_latency": f"{avg_latency:.2f}",
                "total_requests_in_window": total_requests,
                "window_s": alert_metric_store.window_seconds
            }
        )

# --- Reference Solution: LLM Inference Endpoint with Observability ---

# Define OpenTelemetry metrics globally for the application
request_counter = meter.create_counter(
    "llm_requests_total", description="Total number of LLM inference requests", unit="1"
)
request_latency_histogram = meter.create_histogram(
    "llm_request_latency_seconds", description="Latency of LLM inference requests in seconds", unit="s"
)
error_counter = meter.create_counter(
    "llm_errors_total", description="Total number of LLM inference errors", unit="1"
)

@app.post("/generate")
async def generate_text(request: PromptRequest, http_request: Request):
    start_time = time.perf_counter()
    success = False
    response_content = ""
    status_code = status.HTTP_200_OK
    error_message = None

    # Start a new OpenTelemetry span for the entire API request
    with tracer.start_as_current_span("generate_text_api_request") as span:
        span.set_attribute("http.method", http_request.method)
        span.set_attribute("http.target", http_request.url.path)
        span.set_attribute("llm.prompt_length", len(request.prompt))
        span.set_attribute("llm.prompt", request.prompt) # Be cautious with sensitive data in traces

        # Increment total request counter
        request_counter.add(1, {"endpoint": "/generate"})

        try:
            # Call the mock LLM inference function
            response_content = mock_llm_inference(request.prompt)
            success = True
            logger.info(
                "LLM inference successful",
                extra_data={
                    "prompt": request.prompt[:100],
                    "response_preview": response_content[:100],
                    "trace_id": format(span.context.trace_id, '032x')
                }
            )
        except RuntimeError as e:
            error_message = str(e)
            status_code = status.HTTP_500_INTERNAL_SERVER_ERROR
            # Mark the span as errored and add exception details
            span.set_status(trace.Status(trace.StatusCode.ERROR, error_message))
            span.record_exception(e)
            # Increment error counter
            error_counter.add(1, {"endpoint": "/generate"})
            logger.error(
                "LLM inference failed",
                extra_data={
                    "prompt": request.prompt[:100],
                    "error": error_message,
                    "trace_id": format(span.context.trace_id, '032x')
                }
            )
        except Exception as e:
            error_message = f"An unexpected error occurred: {e}"
            status_code = status.HTTP_500_INTERNAL_SERVER_ERROR
            span.set_status(trace.Status(trace.StatusCode.ERROR, error_message))
            span.record_exception(e)
            error_counter.add(1, {"endpoint": "/generate", "error_type": "unexpected"})
            logger.critical(
                "Unexpected error during LLM inference",
                extra_data={
                    "prompt": request.prompt[:100],
                    "error": error_message,
                    "trace_id": format(span.context.trace_id, '032x')
                }
            )

        duration = time.perf_counter() - start_time

        # Record request latency
        request_latency_histogram.record(duration, {"endpoint": "/generate", "success": success})

        # Update the in-memory alerting metric store
        alert_metric_store.record_request(success, duration)

        # Set final span attributes
        span.set_attribute("http.status_code", status_code)
        span.set_attribute("llm.response_length", len(response_content))
        span.set_attribute("llm.request_duration_seconds", duration)

        if not success:
            raise HTTPException(status_code=status_code, detail=error_message)

        return {"response": response_content, "latency_seconds": duration}

# --- Instructions to Run and Test ---
# To run this FastAPI application:
# 1. Save the content of this cell (or the entire notebook) as a Python file, e.g., `main.py`.
# 2. Open your terminal in the same directory.
# 3. Install necessary libraries: `pip install fastapi uvicorn 'opentelemetry-sdk[metrics,trace]' opentelemetry-exporter-otlp-proto-http pydantic`
# 4. Run the application: `uvicorn main:app --port 8000 --reload`
#    (The `--reload` flag is useful for development, automatically restarting the server on code changes).

# To test the API and observe observability output:
# Open another terminal and send requests using `curl`:
# - Successful request:
#   `curl -X POST -H "Content-Type: application/json" -d '{"prompt": "Tell me a story about a space-faring cat."}' http://localhost:8000/generate`
# - Request to trigger errors (you might need to send multiple requests due to `fail_rate=0.1`):
#   `curl -X POST -H "Content-Type: application/json" -d '{"prompt": "Generate a complex technical explanation."}' http://localhost:8000/generate`

# Observe the console output where `uvicorn` is running. You should see:
# - Structured JSON logs for each request and error.
# - OpenTelemetry trace spans (from `ConsoleSpanExporter`).
# - OpenTelemetry metrics exported every 5 seconds (from `ConsoleMetricExporter`).

# To manually check alerts (simulate a monitoring system's periodic check):
# You can call `check_alerts()` directly in a Python interpreter or add a simple endpoint to trigger it.
# For example, after sending a few requests, run `check_alerts()` in a separate script or a new cell in a Jupyter environment.
# To force an alert, you could temporarily increase the `fail_rate` in `mock_llm_inference` to, say, 0.8, and send many requests.
# Example of triggering alerts:
# 1. Run the FastAPI app.
# 2. Send ~10-20 requests quickly, some will fail due to `fail_rate=0.1`.
# 3. In a separate Python session or a new cell in this notebook, run:
#    `from main import check_alerts; check_alerts()`
#    (Assuming you saved this file as `main.py`)
#    You should see an ALERT log if the error rate or average latency exceeds thresholds within the 30-second window.
